# SPY / QQQ Options Quant Dashboard (live)

A live dashboard over the **London Strategic Edge** options feed. It doesn't just list IV and delta. It turns the chain and the tape into positioning, volatility and flow signals:

| Tab | What it tells you |
|---|---|
| **Price & Levels** | Intraday price with the call wall, put wall, gamma flip, max pain and today's expected-move band; the expected-move cone across expiries |
| **Dealer Positioning** | Dealer gamma exposure (GEX) by strike, net GEX re-priced across spot (the gamma flip), vanna and charm exposure, open interest by strike |
| **Volatility** | ATM term structure vs realized vol, smiles for 4 expiries, 25Δ risk reversal and butterfly across expiries, the IV surface, an expiry summary table |
| **Options Flow** | Cumulative call vs put premium, net directional premium every 5 min, premium by strike and by expiry bucket, largest prints, unusual activity (volume ≫ OI) |

Above the tabs: KPI tiles plus a plain-English **market read** built from the numbers.

**How to run:** paste your key in the next cell, then *Run All*. With no key it runs on synthetic **demo data** so you can check that everything works first.

## 1 · Your API key  ← paste it here

In [ ]:
# ▼▼▼ PASTE YOUR LONDON STRATEGIC EDGE API KEY BETWEEN THE QUOTES ▼▼▼
LSE_API_KEY = "PASTE_YOUR_LSE_API_KEY_HERE"
# ▲▲▲ (keys look like  lse_live_xxxxxxxxxxxx ) ▲▲▲

# Optional settings
SYMBOLS = ["SPY", "QQQ"]
REFRESH_SECONDS = 15          # how often flow / price / KPIs update
CHAIN_REFRESH_SECONDS = 60    # how often the full option chain is re-pulled (heavier call)
MAX_DTE = 60                  # furthest expiry pulled into the chain (days)
FLOW_MIN_PREMIUM = 25_000     # only prints >= this $ premium feed the flow panels
USE_WEBSOCKET = True          # real-time spot + live option tape over WebSocket

import os
# Also accepts the key from the environment so it never has to live in the notebook:
LSE_API_KEY = os.environ.get("LSE_API_KEY") or LSE_API_KEY

## 2 · Setup
Finds the dashboard code next to this notebook. If it isn't there (notebook opened on its own, Google Colab, a fresh Python), it installs the code and its libraries from GitHub. Safe to re-run: it only installs what's missing.

In [ ]:
import importlib, importlib.util, pathlib, sys

REPO_ZIP = "https://github.com/tafall77/Options-flow-dashboard/archive/refs/heads/main.zip"
have = lambda mod: importlib.util.find_spec(mod) is not None

# 1. Notebook inside a copy of the repo: use the optionsdash folder next to it (or above it).
for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (folder / "optionsdash" / "__init__.py").exists():
        if str(folder) not in sys.path:
            sys.path.insert(0, str(folder))
        break

# 2. Anything still missing: install the dashboard code + libraries from GitHub.
missing = [m for m in ("optionsdash", "lse", "plotly", "ipywidgets", "anywidget", "scipy", "pandas") if not have(m)]
if missing:
    print("Installing:", ", ".join(missing), "...")
    %pip install -q --disable-pip-version-check {REPO_ZIP}
    importlib.invalidate_caches()

# 3. Google Colab needs its custom widget manager for the live charts.
if "google.colab" in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

print("Setup OK (no kernel restart needed)" if have("optionsdash") else
      "Setup failed: download the whole repo (Code > Download ZIP) and open the notebook from inside it.")

## 3 · Connect

In [ ]:
from optionsdash import make_feed, LiveDashboard

feed = make_feed(LSE_API_KEY, max_dte=MAX_DTE, flow_min_premium=FLOW_MIN_PREMIUM,
                 use_websocket=USE_WEBSOCKET)
print("Data source:", feed.source)

if feed.live:
    # Connection check: pulls three SPY contracts and shows the raw fields the API returns.
    import pandas as pd
    sample = feed.client.options("SPY", max_dte=7, limit=3)
    display(pd.DataFrame(sample))

## 4 · Launch the live dashboard
Use the **SPY / QQQ** toggle to switch underlyings, **Pause** to stop polling, and the dropdown to change the refresh rate. The notebook stays usable while the dashboard runs.

In [ ]:
dash = LiveDashboard(feed, symbols=SYMBOLS, refresh_seconds=REFRESH_SECONDS,
                     chain_refresh_seconds=CHAIN_REFRESH_SECONDS)
dash.show()

## How to read it

**Dealer gamma (GEX).** Street convention: customers are net long puts and net short calls, so dealers are long call gamma and short put gamma. Net GEX is the dollar delta dealers must trade for a 1% move.
* **Positive GEX:** dealers sell rallies and buy dips. Moves mean-revert and price tends to pin to big strikes.
* **Negative GEX:** dealers chase price. Moves extend and ranges widen.
* **Gamma flip:** the spot level where net GEX changes sign, found by re-pricing every contract's gamma across hypothetical spots.
* **Call wall / put wall:** the strikes with the largest call / put gamma, which tend to act as resistance and support.

**Vanna / charm.** How dealer delta changes when IV moves (vanna) or as time passes (charm). They drive the well-known "vol-crush rally" and the drift into the close and into OPEX.

**Volatility.** ATM IV is interpolated at spot from OTM options. The 30d value uses constant-maturity, variance-time interpolation. **VRP** = 30d IV − 20d realized. A **25Δ risk reversal** below zero means puts are bid over calls. The **butterfly** measures wing richness. An **inverted term structure** (7d IV above 30d) flags near-term stress.

**Expected move.** The ATM straddle price for each expiry, falling back to spot × IV × √T when there's no straddle.

**Flow.** Prints at or above `FLOW_MIN_PREMIUM`. Side comes from the quote rule (print vs bid/ask mid) when quotes exist, otherwise from a tick test. *Bullish premium* = calls bought + puts sold − calls sold − puts bought. *Unusual activity* = volume far above open interest, meaning new positions.

**Positioning weight.** Open interest when the feed provides it. Otherwise today's volume, and every chart then says so.

> These are model-based estimates of dealer positioning, not a view of real dealer books. Use them as context, not as trade signals.

## 5 · Stop / extras

In [ ]:
# Stop live updates (the Pause button does the same):
# dash.stop()

# Static render, for front-ends that can't show widgets (e.g. some IDE previews):
# dash.show_static("SPY")

# Save a shareable HTML snapshot of every chart and table for both symbols:
# dash.save_html("options_dashboard.html")

# Raw numbers behind the charts, for your own research:
# snap = dash.snapshots["SPY"]
# snap.metrics            # every KPI as a dict
# snap.term               # per-expiry ATM IV, 25Δ RR / fly, expected move
# snap.exposure           # GEX / vanna / charm / OI by strike
# snap.chain.head()       # normalized chain with model greeks
# snap.flow.tail()        # classified prints